In [4]:
import pandas as pd

In [10]:
df=pd.read_csv('spotify_millsongdata.csv')

In [11]:
df.head(5)

,artist,song,link,text
0,ABBA,Ahe's My Kind Of Girl,/a/abba/ahes+my+kind+of+girl_20598417.html,"Look at her face, it's a wonderful face \r\nA..."
1,ABBA,"Andante, Andante",/a/abba/andante+andante_20002708.html,"Take it easy with me, please \r\nTouch me gen..."
2,ABBA,As Good As New,/a/abba/as+good+as+new_20003033.html,I'll never know why I had to go \r\nWhy I had...
3,ABBA,Bang,/a/abba/bang_20598415.html,Making somebody happy is a question of give an...
4,ABBA,Bang-A-Boomerang,/a/abba/bang+a+boomerang_20002668.html,Making somebody happy is a question of give an...


In [12]:
df.tail(5)

,artist,song,link,text
57645,Ziggy Marley,Good Old Days,/z/ziggy+marley/good+old+days_10198588.html,Irie days come on play \r\nLet the angels fly...
57646,Ziggy Marley,Hand To Mouth,/z/ziggy+marley/hand+to+mouth_20531167.html,Power to the workers \r\nMore power \r\nPowe...
57647,Zwan,Come With Me,/z/zwan/come+with+me_20148981.html,all you need \r\nis something i'll believe \...
57648,Zwan,Desire,/z/zwan/desire_20148986.html,northern star \r\nam i frightened \r\nwhere ...
57649,Zwan,Heartsong,/z/zwan/heartsong_20148991.html,come in \r\nmake yourself at home \r\ni'm a ...


In [13]:
df.shape

(57650, 4)

In [14]:
df.isnull().sum()

artist    0
song      0
link      0
text      0
dtype: int64

In [15]:
df =df.sample(5000).drop('link', axis=1).reset_index(drop=True)

In [16]:
df.head(10)

,artist,song,text
0,Frank Sinatra,Drinking Again,"Drinkin' again and thinkin' of when, when you ..."
1,Johnny Cash,Daddy Sang Bass,I remember when I was a lad \r\nTimes were ha...
2,Widespread Panic,Diner,"Beautiful, the girl's cooking in the diner tod..."
3,The Script,Exit Wounds,My hands are cold \r\nMy body's numb \r\nI'm...
4,Heart,These Dreams,Spare a little candle \r\nSave some light for...
5,Tracy Chapman,Smoke And Ashes,I'd heard rumors and I'd heard talk \r\nAbout...
6,Kelly Family,All Along The Way,You and I are the best of friends I know \r\n...
7,Kinks,Jack The Idiot Dunce,"Who's the fool with the cross-eyed stare, \r\..."
8,Ziggy Marley,Wild And Free,Do you recommend this? \r\nA fire burns for f...
9,Pearl Jam,Beast Of Burden,I will be the least of your burdens \r\nMy ba...


In [17]:
df['text'][0]

"Drinkin' again and thinkin' of when, when you loved me  \r\nI'm havin' a few and wishin' that you were here  \r\n  \r\nMakin' the rounds, accepting a round from strangers  \r\nBein' a fool just hopin' that you'll appear  \r\n  \r\nSure, I can borrow a smoke, maybe tell some joker a bad joke  \r\nBut nobody laughs, they don't laugh at a broken heart  \r\n  \r\nOh, yeah, I'm drinkin' again, it's always the same  \r\nThat same old story  \r\nAfter the kicks there's little old mixed-up me  \r\nTryin' to lose a dream that used to be  \r\n  \r\nLook at me, I'm drinkin' again, drinkin' all over town  \r\nYeah, I'm drinkin' again\r\n\r\n"

In [18]:
df.shape

(5000, 3)

In [19]:
#text cleaning anf preprocessing

In [20]:
import nltk
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()

def tokenization(txt):
    tokens = nltk.word_tokenize(txt)
    stemming = [stemmer.stem(w) for w in tokens]
    return " ".join(stemming)

In [21]:
df['text'] = df['text'].apply(lambda x: tokenization(x))

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:

tfidvector = TfidfVectorizer(analyzer='word',stop_words='english')
matrix = tfidvector.fit_transform(df['text'])
similarity = cosine_similarity(matrix)

In [24]:
similarity[0]

array([1.        , 0.03117441, 0.02585772, ..., 0.02569139, 0.00543855,
       0.0105366 ])

In [25]:
df[df['song'] == 'Crying Over You']

,artist,song,text
621,UB40,Crying Over You,cri over you in the morn cri over you in the e...


In [26]:

def recommendation(song_df):
    idx = df[df['song'] == song_df].index[0]
    distances = sorted(list(enumerate(similarity[idx])),reverse=True,key=lambda x:x[1])
    
    songs = []
    for m_id in distances[1:21]:
        songs.append(df.iloc[m_id[0]].song)
        
    return songs

In [27]:
recommendation('Crying Over You')

['Crying',
 'Cry Cry Cry',
 "Don't Cry Baby",
 "Don't Cry",
 'Cry Over Me',
 'Hope For Love',
 "Isn't Life Strange",
 'The Same Love That Made Me Laugh',
 "Don't Cry Baby",
 "Cry 'til My Tears Run Dry",
 'Crying In The Rain',
 'The Boy In The Bubble',
 'Not A Second Time',
 'Speak To Me Lord',
 'I Saw It In The Mirror',
 'Bird Song',
 'Sonny Cried',
 'Blindman',
 "I'm Sorry",
 "Don't Pass Me By"]

In [28]:
import pickle
pickle.dump(similarity,open('similarity.pkl','wb'))
pickle.dump(df,open('df.pkl','wb'))